# Webinar 4 — Pipeline & Cross Validation
**Dataset: Titanic**

---
1. Cross Validation
2. Pipeline
---

In [7]:
import pandas as pd
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report

df = sns.load_dataset('titanic')[['survived', 'pclass', 'sex', 'fare']].dropna()

X = df.drop('survived', axis=1)
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (712, 3) | Test: (179, 3)


---
## 1. Cross Validation

Cuando evaluamos un modelo con un solo train/test split, el resultado depende de *cómo* cayó ese split — puede ser suerte.

**Cross Validation** repite el proceso `k` veces con distintas particiones y promedia los resultados.

```
Fold 1: [TEST] [train] [train] [train] [train]
Fold 2: [train] [TEST] [train] [train] [train]
Fold 3: [train] [train] [TEST] [train] [train]
Fold 4: [train] [train] [train] [TEST] [train]
Fold 5: [train] [train] [train] [train] [TEST]
```

El árbol de decisión es un modelo que **puede memorizar** los datos de entrenamiento si lo dejamos crecer sin límite. Con CV vamos a ver ese efecto claramente.

In [8]:
# Árbol sin límite — memoriza el train (overfitting)
arbol_libre = DecisionTreeClassifier(random_state=42)

X_encoded = pd.get_dummies(X, columns=['sex'], drop_first=True)

scores_libre = cross_val_score(arbol_libre, X_encoded, y, cv=5, scoring='accuracy')
print("Sin límite de profundidad:")
print("  Accuracy por fold:", scores_libre.round(3))
print(f"  Promedio: {scores_libre.mean():.3f} ± {scores_libre.std():.3f}")

Sin límite de profundidad:
  Accuracy por fold: [0.76  0.792 0.848 0.826 0.837]
  Promedio: 0.813 ± 0.032


In [9]:
# Árbol con max_depth=3 — más generalizable
arbol_limitado = DecisionTreeClassifier(max_depth=3, random_state=42)

scores_limitado = cross_val_score(arbol_limitado, X_encoded, y, cv=5, scoring='accuracy')
print("Con max_depth=3:")
print("  Accuracy por fold:", scores_limitado.round(3))
print(f"  Promedio: {scores_limitado.mean():.3f} ± {scores_limitado.std():.3f}")

Con max_depth=3:
  Accuracy por fold: [0.816 0.803 0.815 0.792 0.815]
  Promedio: 0.808 ± 0.009


In [10]:
# ¿Cuál es la mejor profundidad? CV nos lo dice
for depth in [1, 2, 3, 5, 10, None]:
    arbol = DecisionTreeClassifier(max_depth=depth, random_state=42)
    scores = cross_val_score(arbol, X_encoded, y, cv=5, scoring='accuracy')
    print(f"  max_depth={str(depth):<5}  {scores.mean():.3f} ± {scores.std():.3f}")

  max_depth=1      0.787 ± 0.019
  max_depth=2      0.773 ± 0.019
  max_depth=3      0.808 ± 0.009
  max_depth=5      0.804 ± 0.013
  max_depth=10     0.804 ± 0.030
  max_depth=None   0.813 ± 0.032


El **promedio** nos dice qué tan bueno es el modelo.  
La **desviación** nos dice qué tan estable es entre distintos splits.

Un modelo con `0.80 ± 0.01` es mucho más confiable que uno con `0.80 ± 0.08`.

---
## 2. Pipeline

Un Pipeline encadena pasos de preprocesamiento + modelo en un solo objeto.

**Sin Pipeline** el flujo es manual y fácil de romper:

In [11]:
# ❌ Sin Pipeline
from sklearn.preprocessing import StandardScaler


# Entrenemos un modelo con max_depth=3 sin usar Pipeline, para ver lo que hay que hacer manualmente:

scaler = StandardScaler()
X_train_manual = X_train.copy()
X_test_manual  = X_test.copy()

# Convertir de x categorical a numérica manualmente
X_train_manual['sex'] = (X_train_manual['sex'] == 'male').astype(int)
X_test_manual['sex']  = (X_test_manual['sex']  == 'male').astype(int)

#Escalar las columnas numéricas
X_train_manual[['fare', 'pclass']] = scaler.fit_transform(X_train_manual[['fare', 'pclass']])
X_test_manual[['fare', 'pclass']]  = scaler.transform(X_test_manual[['fare', 'pclass']])

#ahora si entrenamos el árbol con los datos manualmente procesados
arbol = DecisionTreeClassifier(max_depth=3, random_state=42)
arbol.fit(X_train_manual, y_train)
y_pred = arbol.predict(X_test_manual)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.79      0.88      0.83       105
           1       0.79      0.68      0.73        74

    accuracy                           0.79       179
   macro avg       0.79      0.78      0.78       179
weighted avg       0.79      0.79      0.79       179



**Con Pipeline** todo queda encadenado y el fit/transform se maneja automáticamente:

In [12]:
# ✅ Con Pipeline
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), ['fare', 'pclass']),
    ('cat', OneHotEncoder(drop='first'), ['sex']),
])

pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(max_depth=3, random_state=42)),
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.79      0.88      0.83       105
           1       0.79      0.68      0.73        74

    accuracy                           0.79       179
   macro avg       0.79      0.78      0.78       179
weighted avg       0.79      0.79      0.79       179



### Cross Validation con Pipeline

La combinación más poderosa: el pipeline garantiza que el preprocesamiento
ocurra **dentro de cada fold**, sin data leakage.

In [13]:
scores_pipe = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')

print("Accuracy por fold:", scores_pipe.round(3))
print(f"Promedio: {scores_pipe.mean():.3f} ± {scores_pipe.std():.3f}")

Accuracy por fold: [0.816 0.803 0.815 0.792 0.815]
Promedio: 0.808 ± 0.009


---
## 3. GridSearchCV

Antes hacíamos esto a mano para encontrar el mejor `max_depth`:
```python
for depth in [1, 2, 3, 5, 10]:
    ...
```

**GridSearchCV** hace exactamente eso pero de forma automática, probando todas las combinaciones con Cross Validation incluido.

In [14]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__max_depth': [1, 2, 3, 5, 10],
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print("Mejor max_depth:", grid.best_params_)
print(f"Mejor accuracy en CV: {grid.best_score_:.3f}")

Mejor max_depth: {'model__max_depth': 10}
Mejor accuracy en CV: 0.805


---
## Resumen

| Concepto | Para qué sirve |
|---|---|
| **Cross Validation** | Evaluar el modelo de forma robusta, detectar overfitting |
| **Pipeline** | Encadenar preprocesamiento + modelo en un solo objeto |
| **Juntos** | CV con Pipeline garantiza que no haya data leakage en ningún fold |